# Novo on a GPU

Builds Novo with CUDA and checks it on Colab's GPU. Choose **Runtime → Change runtime type → T4 GPU** first, then **Runtime → Run all**. The build takes about 15 minutes; the last cell prints a summary to paste back.

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv

In [ ]:
%%bash
set -e
curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal > /dev/null
source $HOME/.cargo/env
rustc --version
rm -rf novo && git clone -q --depth 1 -b overhaul https://github.com/Simonf8/novo
nvcc --version | tail -1

In [ ]:
%%bash
set -e
source $HOME/.cargo/env
cd novo
export CUDA_COMPUTE_CAP=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1 | tr -d .)
cargo build --release --features cuda 2>&1 | tail -3

In [ ]:
!cd novo && ./target/release/novo run examples/gpu/check.nv cuda 2>&1 | tee ../check.txt

In [ ]:
%%bash
pip -q install huggingface_hub
python -c "from huggingface_hub import snapshot_download; snapshot_download('HuggingFaceTB/SmolLM2-360M-Instruct', local_dir='smollm2')"

In [ ]:
%%bash
cd novo
for run in 'cpu f32' 'cuda f32' 'cuda f16' 'cuda bf16'; do
  echo "== $run"
  ./target/release/novo run examples/llm/generate.nv ../smollm2 'The capital of France is' 64 0.0 $run 2>&1 | tail -2
done | tee ../generate.txt

In [ ]:
!cd novo && git rev-parse --short HEAD && nvidia-smi --query-gpu=name --format=csv,noheader && cat ../check.txt ../generate.txt